In [1]:
import os

# Save the current PATH
original_path = os.environ['PATH']

# Set CUDA 12.5 environment variables, appending the original PATH explicitly
os.environ['CUDA_HOME'] = '/usr/local/cuda-12.5'
os.environ['PATH'] = f"/usr/local/cuda-12.5/bin:{original_path}"
os.environ['LD_LIBRARY_PATH'] = f"/usr/local/cuda-12.5/lib64:{os.environ.get('LD_LIBRARY_PATH', '')}"

#!rm -rf /home/ids/yuhe/.cache/torch_extensions

CODE_DIR = '/home/ids/yuhe/Projects/CA_with_GAN/3_code/diffusion-AE/'
os.chdir(f'{CODE_DIR}')

notebook_path = os.getcwd()

print('Current working directory is:', '\n', notebook_path)

# %load_ext autoreload
# %autoreload 2

Current working directory is: 
 /home/ids/yuhe/Projects/CA_with_GAN/3_code/diffusion-AE


In [2]:
import os
import sys

import shutil
import random
import torch
import torch.optim as optim
import torch.nn.functional as F

import argparse
from pathlib import Path
from PIL import Image

sys.path.append(".")
sys.path.append("..")
from custom_funcs.data_funcs import get_dataloaders
from custom_funcs.model_funcs import load_diffusion_model, load_cs_model
from custom_funcs.eval_funcs import get_fixed_for_test
from custom_funcs.utils import load_hyparams_from_json
from dataset import LMDBDataset
# from custom_funcs.train_funcs import *
# from custom_funcs.loss_funcs import calc_latent_loss
# from custom_funcs.utils import visualize_training_recons

In [3]:
device = "cuda" if torch.cuda.is_available() else "cpu"

#### Load the diffusionAE model

In [4]:
diff_model, conf = load_diffusion_model(device)

Seed set to 0


Model params: 160.69 M


##### Load CS (common and salient) model

In [5]:
args_path = "results/baseline/lr0.01/hyparams.json"
args = load_hyparams_from_json(args_path)
args.cs_model_ckpt = "results/baseline/lr0.01/checkpoints/model_epoch_800.pth"
cs_model=load_cs_model(device, args, is_train=False)

Loaded checkpoint from results/baseline/lr0.01/checkpoints/model_epoch_800.pth


#### Load data

In [6]:
from custom_funcs.data_funcs import config_dataset_path, load_lmdb_dataset
train_path_bg, train_path_t, val_path_bg, val_path_t = config_dataset_path(dataset_type="ffhq")
trainloader_bg, trainloader_t = load_lmdb_dataset(train_path_bg, train_path_t, shuffle=True)
valloader_bg, valloader_t = load_lmdb_dataset(val_path_bg, val_path_t, shuffle=False)

# dataset_bg = LMDBDataset(lmdb_path_bg, load_image=True)
# dataset_t = LMDBDataset(lmdb_path_t, load_image=True)

test_batch_bg, test_batch_t = next(iter(valloader_bg)), next(iter(valloader_t))




In [7]:
# # print(len(dataset_bg))
# # val_loader = DataLoader(dataset_bg, batch_size=4, shuffle=True, num_workers=4)

# # test_batch = next(iter(val_loader))
# from custom_funcs.eval_funcs import run_on_batch, show_diffAE

# for batch_bg in valloader_bg:
#     imgs_bg = batch_bg['img'].to(device)
#     latents_bg = batch_bg['latent'].to(device)
#     break


#     # No image loaded

In [11]:
# from custom_funcs.eval_funcs import show_diffAE
# imgs = test_batch_t['img'][3].to(device)
# latents = test_batch_t['latent'][3].to(device)
# latents.shape

test_image_fixed = get_fixed_for_test(test_batch_bg, test_batch_t, diff_model, device, T_noise=100, T_render=100, idx=1)

original = test_image_fixed["ori_bg"]
latent = test_image_fixed["diffAE_zbg"]
pred = test_image_fixed["pred_diffAE_bg"]

In [12]:
print(original.min().item(), original.max().item())
print("\n")
print(pred.min().item(), pred.max().item())

-0.9450980424880981 1.0


0.018208175897598267 1.0


In [ ]:
xT = diff_model.encode_stochastic(test_image_fixed["ori_bg"], test_image_fixed["diffAE_zbg"], T=100)
pred = diff_model.render(xT, test_image_fixed["diffAE_zbg"], T=100)
from custom_funcs.eval_funcs import show_diffAE

show_diffAE(test_image_fixed["ori_bg"], , xT, idx=0)